# Solución 01 · Construcción del dataset y modelos de ventas


In [1]:
from pathlib import Path
import sys
import warnings
from datetime import datetime

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 160)
pd.set_option("display.width", 160)

def find_project_root(start=None):
    current = Path(start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "datos").exists() and (candidate / "escenarios").exists():
            return candidate
    raise FileNotFoundError("No se ha encontrado la raíz del proyecto FraSoHome.")

PROJECT_ROOT = find_project_root()
DEMO_DIR = PROJECT_ROOT / "escenarios" / "04-analitica-ventas-ml"
ARTIFACT_DIR = DEMO_DIR / "artifacts"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(DEMO_DIR))

from common.frasohome_ml import (
    build_campaign_dataset,
    build_churn_dataset,
    build_return_risk_dataset,
    clean_fact_for_ml,
    load_artifact,
    load_operational_tables,
    model_importance,
    prepare_crm,
    prepare_fact,
    save_artifact,
    score_deciles,
    train_binary_model,
    transformed_feature_names,
)

print("PROJECT_ROOT:", PROJECT_ROOT)


PROJECT_ROOT: C:\repos\frasohome-demos


In [2]:
tables = load_operational_tables(PROJECT_ROOT)
display(pd.DataFrame([{"tabla": k, "filas": v.shape[0], "columnas": v.shape[1]} for k, v in tables.items()]))

fact_typed = prepare_fact(tables["fact"])
fact = clean_fact_for_ml(fact_typed)
crm = prepare_crm(tables["crm"])

display(pd.DataFrame({
    "métrica": ["filas raw", "filas ML", "clientes ML", "productos ML", "min ML", "max ML"],
    "valor": [len(fact_typed), len(fact), fact["customer_id_std"].nunique(), fact["product_id_std"].nunique(), fact["event_dt"].min(), fact["event_dt"].max()],
}))


,tabla,filas,columnas
0,fact,4973,59
1,crm,103,20
2,productos,101,18
3,stock,1940,9
4,contactos,480,23
5,web,9030,22
6,feedback,850,15
7,tickets,560,20
8,costes,1829,21


,métrica,valor
0,filas raw,4973
1,filas ML,1237
2,clientes ML,117
3,productos ML,121
4,min ML,2024-01-02 03:21:03
5,max ML,2025-12-12 17:47:00


In [3]:
churn_df = build_churn_dataset(fact, crm, tables.get("web"), tables.get("feedback"), tables.get("tickets"), max_snapshots=8)
CHURN_TARGET = "label_inactive_90d"
CHURN_EXCLUDED = ["customer_id_std", "snapshot_date", "first_purchase_dt", "last_purchase_dt"]
churn_result = train_binary_model(churn_df, CHURN_TARGET, CHURN_EXCLUDED, model_kind="rf", random_state=42)
display(churn_df[CHURN_TARGET].value_counts(normalize=True).rename("proporción").to_frame())
display(pd.Series(churn_result["metrics"]).to_frame("valor"))
display(model_importance(churn_result["pipeline"], top_n=15))
tmp = churn_result["X_test"].copy()
tmp[CHURN_TARGET] = churn_result["y_test"].values
display(score_deciles(tmp, churn_result["pipeline"].predict_proba(churn_result["X_test"])[:, 1], CHURN_TARGET))


,proporción
label_inactive_90d,
0,0.526846
1,0.473154


,valor
rows,596.000000
target_rate,0.473154
test_roc_auc,0.755515
test_average_precision,0.768213
test_f1,0.619048
test_accuracy,0.677852


,feature,importance,abs_value
0,points,0.062205,0.062205
1,margin_gross,0.035031,0.035031
2,avg_order_value,0.033205,0.033205
3,avg_sla_hours,0.031737,0.031737
4,ciudad_Córdoba,0.029915,0.029915
5,avg_unit_price,0.028762,0.028762
6,pct_high_priority_tickets,0.028633,0.028633
7,gross_margin_rate,0.028455,0.028455
8,monetary_gross,0.024707,0.024707
9,revenue_cat_muebles,0.023961,0.023961


,decile,n,avg_score,target_rate
9,10,15,0.813768,0.933333
8,9,15,0.648768,0.733333
7,8,15,0.565116,0.733333
6,7,15,0.509373,0.400000
5,6,14,0.448469,0.500000
4,5,15,0.409957,0.466667
3,4,15,0.371650,0.266667
2,3,15,0.326856,0.266667
1,2,15,0.263757,0.000000
0,1,15,0.188664,0.400000


In [4]:
campaign_df = build_campaign_dataset(tables["contactos"], fact, crm, tables.get("web"), tables.get("feedback"), tables.get("tickets"))
CAMPAIGN_TARGET = "conversion_30d_flag"
CAMPAIGN_EXCLUDED = [
    "contact_id", "campaign_id", "customer_id", "customer_id_std", "fecha_contacto", "contact_dt",
    "conversion_30d_flag", "order_id_conversion_30d", "ticket_id_conversion_30d", "fecha_conversion_30d",
    "revenue_30d_eur", "margen_30d_eur", "first_purchase_dt", "last_purchase_dt",
]
campaign_result = train_binary_model(campaign_df, CAMPAIGN_TARGET, CAMPAIGN_EXCLUDED, model_kind="rf", random_state=42)
display(campaign_df[CAMPAIGN_TARGET].value_counts(normalize=True).rename("proporción").to_frame())
display(pd.Series(campaign_result["metrics"]).to_frame("valor"))
display(model_importance(campaign_result["pipeline"], top_n=15))
tmp = campaign_result["X_test"].copy()
tmp[CAMPAIGN_TARGET] = campaign_result["y_test"].values
display(score_deciles(tmp, campaign_result["pipeline"].predict_proba(campaign_result["X_test"])[:, 1], CAMPAIGN_TARGET))


,proporción
conversion_30d_flag,
0,0.632911
1,0.367089


,valor
rows,158.000000
target_rate,0.367089
test_roc_auc,0.688000
test_average_precision,0.646215
test_f1,0.451613
test_accuracy,0.575000


,feature,importance,abs_value
0,customer_tenure_window_days,0.058811,0.058811
1,revenue_channel_pos,0.056124,0.056124
2,margin_gross,0.033939,0.033939
3,gross_margin_rate,0.031391,0.031391
4,frequency_docs,0.031252,0.031252
5,avg_unit_price,0.026211,0.026211
6,avg_nps,0.025555,0.025555
7,monetary_gross,0.024387,0.024387
8,revenue_cat_textil_hogar,0.024125,0.024125
9,units_sold,0.019921,0.019921


,decile,n,avg_score,target_rate
9,10,4,0.692004,1.00
8,9,4,0.613925,0.50
7,8,4,0.545814,0.00
6,7,4,0.522159,0.25
5,6,4,0.478068,0.50
4,5,4,0.428589,0.25
3,4,4,0.364254,1.00
2,3,4,0.328565,0.00
1,2,4,0.278712,0.25
0,1,4,0.216948,0.00


In [5]:
artifact = {
    "created_at": datetime.now().isoformat(timespec="seconds"),
    "source": "Solución alumnos · CSV operativos FraSoHome 2024-2025",
    "churn": {"dataset": churn_df, "target": CHURN_TARGET, "excluded": CHURN_EXCLUDED, "result": churn_result},
    "campaign": {"dataset": campaign_df, "target": CAMPAIGN_TARGET, "excluded": CAMPAIGN_EXCLUDED, "result": campaign_result},
}
path = save_artifact(artifact, ARTIFACT_DIR / "alumnos_sales_models_artifact.joblib")
print("Guardado:", path)


Guardado: C:\repos\frasohome-demos\escenarios\04-analitica-ventas-ml\artifacts\alumnos_sales_models_artifact.joblib
